# 00 - Setup do ambiente no Google Colab

Este notebook prepara o runtime do Colab para executar os experimentos do projeto:
1. Verifica a GPU (Tesla T4 ou superior),
2. Clona o repositório,
3. Instala as dependências (`requirements-colab.txt`),
4. Carrega o dataset ToLD-BR via Hugging Face (sem Google Drive),
5. Fornece uma célula para baixar resultados ao final da sessão,

## 1. Verificar GPU

In [16]:
!nvidia-smi

Sun Nov 15 15:32:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [17]:
import torch

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))
    print("Versão do PyTorch: ", torch.__version__)
    print("CUDA: ", torch.version.cuda)

CUDA disponível: True
GPU:  Tesla T4
Versão do PyTorch:  2.14.1+cu130
CUDA:  13.0


## 2. Clonar o Repósitório

In [18]:
import os

REPO_URL = "https://github.com/elianisaac/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais.git"
REPO_DIR = "Classifica-o-de-Discurso-T-xico-em-Redes-Sociais"

# Só clona se a pasta NÃO existir
if not os.path.exists(f"/content/{REPO_DIR}"):
    !git clone {REPO_URL} /content/{REPO_DIR}

# Força a entrada no caminho absoluto correto
%cd /content/{REPO_DIR}

/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais


## 3. Instalar Dependencias

Usa `requirements-colab.txt`


In [ ]:
# Atualiza as ferramentas de build do Python
!pip install --upgrade pip setuptools wheel

# Instala as dependencias
%pip install -q -r requirements-collab.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-colab.txt'


In [20]:
# Verificação rápida das versões instaladas
import datasets, transformers, sklearn
print("datasets:", datasets.__version__)
print("transformers: ", transformers.__version__)
print("scikit-learn: ", sklearn.__version__)

datasets: 4.8.5
transformers:  5.22.1
scikit-learn:  1.6.1


# 4. Carregamento do Dataset

O script `src/load_data.py` baixa o dataset do Hugging Face (`JAugusto97/told-br`) e salva em disco (`data/processed/`) para evitar re-executar o *loading script* a cada uso.

In [21]:
!python src/load_data.py --variant binary --output data/processed/ --force

[2026-11-15 15:32:40] [INFO] [load_data] Processando variante: binary
[2026-11-15 15:32:40] [INFO] [load_data] Baixando dados brutos do GitHub: https://raw.githubusercontent.com/JAugusto97/ToLD-BR/main/ToLD-BR.csv
Traceback (most recent call last):
  File "/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/src/load_data.py", line 106, in <module>
    main()
    ~~~~^^
  File "/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/src/load_data.py", line 100, in main
    load_variant(variant, output_dir, force=args.force)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/src/load_data.py", line 58, in load_variant
    df = df[['text', 'toxic']].rename(columns={'toxic': 'label'})
         ~~^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/frame.py", line 4108, in __getitem__
    indexer = self.columns._get_indexer_strict(key, "columns")[1]
              ~~~~~~~~~~~~~~~~~~~~~~~

In [22]:
# Inspeção rápida do dataset carregado
from datasets import load_from_disk
ds = load_from_disk("data/processed/told-br-binary")
print(ds)
print("Exemplo: ", ds["train"][0])

FileNotFoundError: Directory data/processed/told-br-binary not found

In [ ]:
# Roda o pré-processamento gerando os textos limpos
!python src/preprocessing.py \
    --input data/processed/told-br-binary \
    --output data/processed/told-br-binary-clean

[2026-11-14 15:11:59] [INFO] [preprocessing] Carregando dataset de: data/processed/told-br-binary
[2026-11-14 15:11:59] [INFO] [preprocessing] Aplicando pré-processamento nas partições...
[2026-11-14 15:11:59] [INFO] [preprocessing]   Processando split: train
[2026-11-14 15:11:59] [INFO] [preprocessing]   Processando split: test
[2026-11-14 15:11:59] [INFO] [preprocessing]   Processando split: validation
[2026-11-14 15:11:59] [INFO] [preprocessing] Salvando dataset processado em: data/processed/told-br-binary-clean
Saving the dataset (1/1 shards): 100% 16800/16800 [00:00<00:00, 1103297.59 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 393409.19 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 428239.91 examples/s]
[2026-11-14 15:11:59] [INFO] [preprocessing] ============================================================
[2026-11-14 15:11:59] [INFO] [preprocessing] Resumo do Pré-processamento:
[2026-11-14 15:11:59] [INFO] [preprocessing]

## 5. Executar os experimentos

Com o ambiente pronto e o dataset carregado, rode os scripts de treino. Exemplos:"

In [ ]:
# Baseline: Regressão Logística
!python src/train_logistic.py --data data/processed/told-br-binary-clean --output results/models/

[2026-11-14 15:12:05] [INFO] [train_logistic] Carregando dados de: data/processed/told-br-binary-clean
[2026-11-14 15:12:05] [INFO] [train_logistic] Treino: 16800 amostras | Teste: 2100 amostras
[2026-11-14 15:12:05] [INFO] [train_logistic] Iniciando GridSearch (isso pode levar alguns minutos)...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[2026-11-14 15:12:37] [INFO] [train_logistic] Melhores parâmetros: {'clf__C': 1.0, 'clf__class_weight': 'balanced', 'clf__solver': 'liblinear'}
[2026-11-14 15:12:37] [INFO] [train_logistic] Melhor F1 (validação cruzada): 0.7070
[2026-11-14 15:12:38] [INFO] [train_logistic] 
=== Resultados no Teste ===
          f1: 0.6960
   precision: 0.6996
      recall: 0.6924
    accuracy: 0.7200
         auc: 0.8104
[2026-11-14 15:12:38] [INFO] [train_logistic] Modelo salvo em: results/models/logistic_regression_best.joblib
[2026-11-14 15:12:38] [INFO] [train_logistic] Concluído com sucesso.


In [ ]:
# Rodando o BERTimbau
!python src/train_bertimbau.py \
    --data data/processed/told-br-binary-clean \
    --epochs 3 \
    --batch-size 16 \
    --output-dir results/models/bertimbau

In [ ]:
# Avaliação e comparação
!python src/evaluate.py --metrics-dir results/metrics --output-dir results/figures

[2026-11-14 15:12:42] [INFO] [evaluate] Buscando resultados em: results/metrics
[2026-11-14 15:12:42] [INFO] [evaluate] 
Tabela Comparativa:
| Modelo              |   F1-Score |   Precisão |   Recall |   Acurácia |
|:--------------------|-----------:|-----------:|---------:|-----------:|
| Regressão Logística |   0.695967 |   0.699584 | 0.692387 |       0.72 |
[2026-11-14 15:12:42] [INFO] [evaluate] Gerando gráfico comparativo...
/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais/src/evaluate.py:93: UserWarning: The palette list has more values (2) than needed (1), which may not be intended.
  ax = sns.barplot(data=df_melted, x="Métrica", y="Score", hue="Modelo", palette=["#4C72B0", "#C44E52"])
[2026-11-14 15:12:43] [INFO] [evaluate] Gráfico salvo em: results/figures/comparacao_modelos.png
[2026-11-14 15:12:43] [INFO] [evaluate] Tabela CSV salva em: results/metrics/comparacao_final.csv
